<p align="center">
  <img src=".github/images/banners/setup.png" alt="Set up once: sign in, connect a Foundry project, write .env" width="900" />
</p>

# Set up once, then run any capsule

> **What do we need before any capsule notebook will run?**

⏱️ About 10 minutes · 🧱 We'll build: a `.env` file that every capsule in this repo reads

**By the end, we'll be able to:**

- Confirm our Azure sign-in and pick a Microsoft Foundry project
- Write a `.env` file with the project's endpoint and key — without copying secrets by hand
- Read an env-check failure and know exactly what to fix

> 💡 **How to use this notebook:** run each cell in order with **Shift + Enter**. Before each code cell, read the note and **take a guess** at what it will print. Guessing wrong is how the lessons stick.

<img src=".github/images/divider.svg" alt="" width="100%" height="6" />


## 1. What do we need before we start?

| We need | How to check |
|---|---|
| An **Azure subscription** with access to Microsoft Foundry | We can sign in at [ai.azure.com](https://ai.azure.com) |
| **This repo in GitHub Codespaces** (or the devcontainer) | Python, the Azure CLI, and `requirements-dev.txt` are already installed |
| An **Azure sign-in** | `az login --use-device-code` in a terminal completes without errors |

Running locally works too, with Python 3.10+ and the [Azure CLI](https://learn.microsoft.com/en-us/cli/azure/install-azure-cli).

<img src=".github/images/divider.svg" alt="" width="100%" height="6" />


## 2. Are we signed in to Azure?

Every capsule authenticates through our Azure identity, so this is the first thing to confirm. If the cell fails, run `az login --use-device-code` in a terminal and run it again.

> 🔮 **Guess first:** which subscription name will it print — the one we expect, or a different default?

In [ ]:
import json
import subprocess

result = subprocess.run(["az", "account", "show", "--output", "json"], capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError("Not signed in. Run `az login --use-device-code` in a terminal, then re-run this cell.")
account = json.loads(result.stdout)
print(f"✅ Signed in to subscription: {account['name']}")

<img src=".github/images/divider.svg" alt="" width="100%" height="6" />


## 3. Which Foundry project will we use?

We can reuse a project we already have, or create a new one just for these capsules.

- **Reuse** — in [ai.azure.com](https://ai.azure.com), open the project and note its **resource group** and **project name**.
- **Create** — pick a resource group, a project name, and a region. Section 4 creates both if they don't exist. See [Create a project in the Foundry portal](https://learn.microsoft.com/en-us/azure/foundry/how-to/create-projects).

Fill in the values below. ✏️

In [ ]:
MODE = "use"              # ✏️ "use" an existing project, or "create" a new one
RESOURCE_GROUP = ""       # ✏️ e.g. "rg-model-releases"
PROJECT_NAME = ""         # ✏️ e.g. "model-releases"
LOCATION = "eastus2"      # ✏️ only used when MODE == "create"

<img src=".github/images/divider.svg" alt="" width="100%" height="6" />


## 4. How do we get the project's details into `.env`?

The next cell runs [`1-scripts/setenv.sh`](1-scripts/setenv.sh). It copies [`1-scripts/sample.env`](1-scripts/sample.env) to `.env` at the repo root, then asks Azure for the project's endpoint and key and writes them in — so we never paste a secret by hand. It won't overwrite an existing `.env`; delete it first to start over. The `.env` file is git-ignored.

> 🔮 **Guess first:** which lines in `.env` will the script fill in, and which will it leave for us?

In [ ]:
if MODE not in ("use", "create"):
    raise ValueError('MODE must be "use" or "create".')
if not RESOURCE_GROUP or not PROJECT_NAME:
    raise ValueError("Set RESOURCE_GROUP and PROJECT_NAME in the cell above.")

args = ["bash", "1-scripts/setenv.sh", f"--{MODE}", RESOURCE_GROUP, PROJECT_NAME]
if MODE == "create":
    args.append(LOCATION)
subprocess.run(args, check=True)

<img src=".github/images/divider.svg" alt="" width="100%" height="6" />


## 5. Is our environment ready?

Every capsule opens with a check like this one. When it prints `Ready`, any capsule can find our project.

In [ ]:
import os

from dotenv import load_dotenv

load_dotenv(override=True)


def check_env(required):
    missing = [name for name in required if not os.getenv(name)]
    if missing:
        raise EnvironmentError(f"Missing {missing} in .env. Re-run section 4, or add them by hand.")
    return "✅ Ready: " + os.environ["MICROSOFT_FOUNDRY_ENDPOINT"]


print(check_env(["MICROSOFT_FOUNDRY_ENDPOINT", "MICROSOFT_FOUNDRY_API_KEY"]))

### 🔧 Break it on purpose

What does a capsule show us when a setting is missing? Let's ask for one that doesn't exist and read the message — it's the same one we'll see if a capsule needs a deployment we haven't set yet.

> 🔮 **Guess first:** will the message tell us *which* setting is missing, or just that something failed?

In [ ]:
try:
    check_env(["MICROSOFT_FOUNDRY_ENDPOINT", "AZURE_NOT_A_REAL_DEPLOYMENT"])
except EnvironmentError as err:
    print("🔧", err)

<img src=".github/images/divider.svg" alt="" width="100%" height="6" />


## 6. How do we deploy the models a capsule needs?

Each capsule's first cell lists the deployments and extra settings it needs. To deploy a model:

1. In the project, open **Model catalog**, select the model, then **Deploy**.
2. Use the deployment name the capsule expects — the defaults in `.env` match the model name — or update `.env` to match ours.

Some models, such as MAI-Transcribe and MAI-Voice, need no deployment: the capsule names the model in the request. See [Deploy models in Microsoft Foundry](https://learn.microsoft.com/en-us/azure/foundry/how-to/deploy-models-openai).

<img src=".github/images/divider.svg" alt="" width="100%" height="6" />


## 7. What if something fails?

| What we see | What it usually means | What to try |
|---|---|---|
| `Not signed in` | The Azure CLI has no active account | `az login --use-device-code`, then re-run section 2 |
| `Missing [...] in .env` | A setting isn't in `.env` yet | Re-run section 4, or add the listed names by hand |
| `401` / `403` | Wrong key, or our identity lacks the *Cognitive Services User* role | See [Foundry RBAC](https://learn.microsoft.com/en-us/azure/foundry/concepts/rbac-azure-ai-foundry) |
| `404` model not found | The deployment name in `.env` doesn't match the portal | Match it exactly — names are case-sensitive |
| Quota or region errors | The model isn't available in our region, or quota is used up | See [Manage quotas](https://learn.microsoft.com/en-us/azure/foundry/how-to/quota) |

Still stuck? [Open an issue](https://github.com/microsoft-foundry/model-releases/issues/new) with the failing cell's output.

<img src=".github/images/divider.svg" alt="" width="100%" height="6" />


## 8. Where do we go next?

➡️ **Next:** pick a capsule from [What's new?](README.md#whats-new) or the [CHANGELOG](CHANGELOG.md), open its notebook, and run it top to bottom.

**References**

- [Quickstart: Get started with Microsoft Foundry](https://learn.microsoft.com/en-us/azure/foundry/quickstarts/get-started-code)
- [Create a project in the Foundry portal](https://learn.microsoft.com/en-us/azure/foundry/how-to/create-projects)
- [Microsoft Foundry Models overview](https://learn.microsoft.com/en-us/azure/foundry/concepts/foundry-models-overview)